In [2]:
import pandas as pd

## Load The Dataset

In [5]:
sd = pd.read_csv('transactions.csv')
sub = pd.read_csv('subscriptions.csv')

In [6]:
sd = pd.read_csv("transactions.csv")
sub = pd.read_csv("subscriptions.csv")

print("Transactions Shape:", sd.shape)
print("Subscriptions Shape:", sub.shape)

Transactions Shape: (213341, 27)
Subscriptions Shape: (674, 12)


In [7]:
# Dataset Columns
print("Transactions Columns:")
print(sd.columns.tolist())

print("\nSubscriptions Columns:")
print(sub.columns.tolist())

# Missing Values
print("\nTransactions Missing Values:")
print(sd.isnull().sum()[sd.isnull().sum() > 0])

print("\nSubscriptions Missing Values:")
print(sub.isnull().sum()[sub.isnull().sum() > 0])

# Duplicate Rows
print("\nTransactions Duplicates:", sd.duplicated().sum())
print("Subscriptions Duplicates:", sub.duplicated().sum())

Transactions Columns:
['transaction_id', 'account_id', 'customer_id', 'timestamp', 'amount', 'currency', 'merchant_name', 'merchant_category', 'merchant_mcc', 'transaction_type', 'payment_channel', 'city', 'country', 'counterparty_iban', 'label', 'category', 'subcategory', 'is_subscription', 'is_salary', 'is_transfer', 'is_cash_withdrawal', 'balance_after_transaction', 'is_foreign', 'original_amount', 'original_currency', 'fx_rate', 'foreign_fee']

Subscriptions Columns:
['subscription_id', 'customer_id', 'account_id', 'merchant_name', 'category', 'subcategory', 'amount', 'frequency_months', 'billing_day', 'start_date', 'end_date', 'status']

Transactions Missing Values:
city                  99110
counterparty_iban    209343
dtype: int64

Subscriptions Missing Values:
end_date    564
dtype: int64

Transactions Duplicates: 0
Subscriptions Duplicates: 0


## Transactions Dataset Cleaning

In [8]:
sd.drop(
    columns=['city', 'counterparty_iban'],
    inplace=True
)

print("Dataset Shape:", sd.shape)
print("\nMissing Values:")
print(sd.isnull().sum().sum())

Dataset Shape: (213341, 25)

Missing Values:
0


## Subscriptions Dataset Check

In [9]:
print("Subscription Status:")
print(sub['status'].value_counts())

print("\nEnd Date Missing by Status:")
print(
    pd.crosstab(
        sub['status'],
        sub['end_date'].isnull(),
        margins=True
    )
)

Subscription Status:
status
active     572
churned    102
Name: count, dtype: int64

End Date Missing by Status:
end_date  False  True  All
status                    
active        8   564  572
churned     102     0  102
All         110   564  674


# Investigate 8 Active Subscriptions

In [10]:
active_end = sub[
    (sub['status'] == 'active') &
    (sub['end_date'].notna())
]

print(active_end[
    ['subscription_id', 'merchant_name',
     'start_date', 'end_date', 'status']
].to_string(index=False))

subscription_id    merchant_name start_date   end_date status
      S00000221 Childcare Center 2025-03-16 2026-10-14 active
      S00000329 Childcare Center 2024-04-30 2026-10-14 active
      S00000477 Childcare Center 2025-11-25 2026-10-14 active
      S00000553 Childcare Center 2024-04-25 2026-10-14 active
      S00000561 Childcare Center 2026-05-23 2026-10-14 active
      S00000601 Childcare Center 2024-12-02 2026-10-14 active
      S00000628 Childcare Center 2025-03-20 2026-10-14 active
      S00000642 Childcare Center 2026-08-26 2026-10-14 active


## Target Variable Analysis

In [11]:
print("Total Transactions:", len(sd))

print("\nSubscription Distribution:")
print(sd['is_subscription'].value_counts())

print("\nSubscription Percentage:")
print(
    sd['is_subscription'].value_counts(normalize=True) * 100
)

print("\nUnique Customers:")
print(sd['customer_id'].nunique())

print("\nUnique Merchants:")
print(sd['merchant_name'].nunique())

Total Transactions: 213341

Subscription Distribution:
is_subscription
False    205997
True       7344
Name: count, dtype: int64

Subscription Percentage:
is_subscription
False    96.557624
True      3.442376
Name: proportion, dtype: float64

Unique Customers:
100

Unique Merchants:
180


## Subscription Merchant Analysis

In [12]:
# Subscription Transactions
subs = sd[sd['is_subscription'] == True].copy()

# Top 15 Subscription Merchants
print("Top 15 Subscription Merchants:")

print(
    subs['merchant_name']
    .value_counts()
    .head(15)
)

# Customer-Merchant Recurring Groups
groups = subs.groupby(
    ['customer_id', 'merchant_name']
).size()

print("\nTotal Customer-Merchant Groups:", len(groups))

print("\nGroups with 1 Payment:", (groups == 1).sum())

print("Groups with 2+ Payments:", (groups >= 2).sum())

print("Groups with 3+ Payments:", (groups >= 3).sum())

print("\nMaximum Payments in One Group:", groups.max())

Top 15 Subscription Merchants:
merchant_name
ConEdison        643
Verizon          579
AT&T             576
Amazon Prime     517
National Grid    489
State Farm       485
Geico Auto       449
Xfinity          434
T-Mobile         429
Netflix          388
Disney+          364
Spotify          362
Blue Cross       358
Apple iCloud+    304
Hulu             207
Name: count, dtype: int64

Total Customer-Merchant Groups: 518

Groups with 1 Payment: 42
Groups with 2+ Payments: 476
Groups with 3+ Payments: 442

Maximum Payments in One Group: 36


## Recurring Payment Interval Analysis

In [13]:
# Convert timestamp
sd['timestamp'] = pd.to_datetime(sd['timestamp'])

# Select subscription transactions
subs = sd[sd['is_subscription'] == True].copy()

# Sort transactions
subs = subs.sort_values(
    ['customer_id', 'account_id', 'merchant_name', 'timestamp']
)

# Calculate payment intervals
subs['payment_interval'] = (
    subs.groupby(
        ['customer_id', 'account_id', 'merchant_name']
    )['timestamp']
    .diff()
    .dt.total_seconds() / 86400
)

# Remove missing intervals
intervals = subs['payment_interval'].dropna()

print("Total Valid Intervals:", len(intervals))

print("\nAverage Interval:", round(intervals.mean(), 2))
print("Median Interval:", round(intervals.median(), 2))

print("\nPayment Frequency Analysis:")

print("Weekly (6-8 days):",
      intervals.between(6, 8).sum())

print("Monthly (27-32 days):",
      intervals.between(27, 32).sum())

print("Quarterly (85-95 days):",
      intervals.between(85, 95).sum())

print("Yearly (350-380 days):",
      intervals.between(350, 380).sum())


Total Valid Intervals: 6825

Average Interval: 37.57
Median Interval: 30.81

Payment Frequency Analysis:
Weekly (6-8 days): 0
Monthly (27-32 days): 6100
Quarterly (85-95 days): 213
Yearly (350-380 days): 5


## Historical Features

In [14]:
# Sort transactions
sd = sd.sort_values(
    ['customer_id', 'account_id', 'merchant_name', 'timestamp']
).reset_index(drop=True)

# Group transactions
group = sd.groupby(
    ['customer_id', 'account_id', 'merchant_name']
)

# Previous payment date
sd['previous_payment_date'] = group['timestamp'].shift(1)

# Days between payments
sd['days_since_last_payment'] = (
    sd['timestamp'] - sd['previous_payment_date']
).dt.total_seconds() / 86400

# Number of previous payments
sd['prior_merchant_payments'] = group.cumcount()

# Previous payment amount
sd['previous_amount'] = group['amount'].shift(1)

# Amount difference
sd['amount_difference'] = (
    sd['amount'].abs() - sd['previous_amount'].abs()
).abs()

# Check new features
print(sd[[
    'merchant_name',
    'days_since_last_payment',
    'prior_merchant_payments',
    'amount_difference',
    'is_subscription'
]].head(15))

   merchant_name  days_since_last_payment  prior_merchant_payments  \
0       7-Eleven                      NaN                        0   
1       7-Eleven                 6.657199                        1   
2       7-Eleven                 8.822766                        2   
3       7-Eleven                 4.089051                        3   
4       7-Eleven                12.198819                        4   
5       7-Eleven                 6.915880                        5   
6       7-Eleven                10.722488                        6   
7       7-Eleven                21.931343                        7   
8       7-Eleven                 0.393600                        8   
9       7-Eleven                 2.747859                        9   
10      7-Eleven                 0.894641                       10   
11      7-Eleven                25.402488                       11   
12      7-Eleven                 3.900613                       12   
13      7-Eleven    

## Subscription vs Non-Subscription Feature Comparison

In [ ]:
# Compare Subscription vs Normal Transactions

features = [
    'days_since_last_payment',
    'prior_merchant_payments',
    'amount_difference'
]

print("Feature Comparison:")

print(sd.groupby('is_subscription')[features].median().round(2))

# Monthly payment pattern
sd['monthly_pattern'] = (sd['days_since_last_payment'].between(27, 32))
# Similar payment amountsd['similar_amount'] = (sd['amount_difference'] <= 2)
print("\nMonthly Pattern Percentage:")
print(sd.groupby('is_subscription')['monthly_pattern'].mean() * 100)
print("\nSimilar Amount Percentage:")
print(sd.groupby('is_subscription')['similar_amount'].mean() * 100)

Feature Comparison:
                 days_since_last_payment  prior_merchant_payments  \
is_subscription                                                     
False                              15.06                     20.0   
True                               30.74                     10.0   

                 amount_difference  
is_subscription                     
False                        13.49  
True                          0.00  

Monthly Pattern Percentage:
is_subscription
False    10.447725
True     77.355664
Name: monthly_pattern, dtype: float64

Similar Amount Percentage:
is_subscription
False    16.131788
True     85.049020
Name: similar_amount, dtype: float64


## Combined Pattern Analysis

In [17]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# Combined rule
sd['rule_subscription'] = (sd['monthly_pattern'] &sd['similar_amount'] &(sd['prior_merchant_payments'] >= 1))
# Compare with actual labels
print("Combined Pattern Distribution:")

print(pd.crosstab(sd['is_subscription'],sd['rule_subscription'],margins=True))

# Evaluate rule-based detection
print("\nClassification Report:")

print(classification_report(sd['is_subscription'],sd['rule_subscription'],zero_division=0))

print("\nConfusion Matrix:")

print(confusion_matrix( sd['is_subscription'], sd['rule_subscription']))

Combined Pattern Distribution:
rule_subscription   False  True     All
is_subscription                        
False              202137  3860  205997
True                 1886  5458    7344
All                204023  9318  213341

Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.98      0.99    205997
        True       0.59      0.74      0.66      7344

    accuracy                           0.97    213341
   macro avg       0.79      0.86      0.82    213341
weighted avg       0.98      0.97      0.97    213341


Confusion Matrix:
[[202137   3860]
 [  1886   5458]]


## Payment Interval Consistency

In [18]:
# Previous payment interval
group = sd.groupby(
    ['customer_id', 'account_id', 'merchant_name']
)

sd['previous_interval'] = (
    group['days_since_last_payment'].shift(1)
)

# Difference between consecutive intervals
sd['interval_consistency'] = (
    sd['days_since_last_payment'] -
    sd['previous_interval']
).abs()

# Quarterly recurring pattern
sd['quarterly_pattern'] = (
    sd['days_since_last_payment'].between(85, 95)
)

# Yearly recurring pattern
sd['yearly_pattern'] = (
    sd['days_since_last_payment'].between(350, 380)
)

# Compare features
print("Interval Consistency Comparison:")

print(
    sd.groupby('is_subscription')[
        ['interval_consistency', 'previous_interval']
    ].median().round(2)
)

print("\nQuarterly Pattern Percentage:")

print(
    sd.groupby('is_subscription')['quarterly_pattern']
    .mean() * 100
)

print("\nYearly Pattern Percentage:")

print(
    sd.groupby('is_subscription')['yearly_pattern']
    .mean() * 100
)

Interval Consistency Comparison:
                 interval_consistency  previous_interval
is_subscription                                         
False                           14.25              14.76
True                             1.38              30.67

Quarterly Pattern Percentage:
is_subscription
False    1.762647
True     2.709695
Name: quarterly_pattern, dtype: float64

Yearly Pattern Percentage:
is_subscription
False    0.065535
True     0.068083
Name: yearly_pattern, dtype: float64


## Target Leakage Check

In [20]:
check_cols = ['label', 'category', 'subcategory']

for col in check_cols:

    print(f"\n===== {col.upper()} =====")

    result = pd.crosstab(sd[col],sd['is_subscription'])

    result['Total'] = result.sum(axis=1)

    result['Subscription_Rate'] = (result.get(True, 0) / result['Total'] * 100)

    print(result.sort_values('Subscription_Rate',ascending=False).head(15))


===== LABEL =====
is_subscription   False  True  Total  Subscription_Rate
label                                                  
CHILDCARE CENTER      0   111    111         100.000000
BLUE CROSS            0   358    358         100.000000
STATE FARM            0   485    485         100.000000
T-MOBILE              0   429    429         100.000000
CITY WATER DEPT       0   127    127         100.000000
CONEDISON             0   643    643         100.000000
GEICO AUTO            0   449    449         100.000000
GITHUB                0    30     30         100.000000
GOOGLE WORKSPACE      0    83     83         100.000000
NATIONAL GRID         0   489    489         100.000000
XFINITY               0   434    434         100.000000
VERIZON               0   579    579         100.000000
ADOBE                 0    84     84         100.000000
AT&T                  0   576    576         100.000000
DISNEY+             499   364    863          42.178447

===== CATEGORY =====
is_subs

## Final Feature Selection

In [21]:
# Selected features
features = [
    'amount',
    'days_since_last_payment',
    'prior_merchant_payments',
    'amount_difference',
    'previous_interval',
    'interval_consistency',
    'monthly_pattern',
    'similar_amount',
    'quarterly_pattern',
    'yearly_pattern'
]

# Create ML dataset
model_df = sd[['timestamp', 'is_subscription'] + features].copy()

# Convert amount to positive magnitude
model_df['amount'] = model_df['amount'].abs()

# Convert boolean features to integers
bool_cols = [
    'monthly_pattern',
    'similar_amount',
    'quarterly_pattern',
    'yearly_pattern'
]

model_df[bool_cols] = model_df[bool_cols].astype(int)

# Handle missing historical features
missing_cols = [
    'days_since_last_payment',
    'amount_difference',
    'previous_interval',
    'interval_consistency'
]

model_df[missing_cols] = model_df[missing_cols].fillna(-1)

# Check final ML dataset
print("ML Dataset Shape:", model_df.shape)

print("\nMissing Values:")
print(model_df.isnull().sum())

print("\nTarget Distribution:")
print(model_df['is_subscription'].value_counts())

print("\nSelected Features:")
print(features)

ML Dataset Shape: (213341, 12)

Missing Values:
timestamp                  0
is_subscription            0
amount                     0
days_since_last_payment    0
prior_merchant_payments    0
amount_difference          0
previous_interval          0
interval_consistency       0
monthly_pattern            0
similar_amount             0
quarterly_pattern          0
yearly_pattern             0
dtype: int64

Target Distribution:
is_subscription
False    205997
True       7344
Name: count, dtype: int64

Selected Features:
['amount', 'days_since_last_payment', 'prior_merchant_payments', 'amount_difference', 'previous_interval', 'interval_consistency', 'monthly_pattern', 'similar_amount', 'quarterly_pattern', 'yearly_pattern']


## Time-Based Data Split

In [22]:
# Sort by time
model_df['timestamp'] = pd.to_datetime(model_df['timestamp'])

model_df = model_df.sort_values(
    'timestamp'
).reset_index(drop=True)

# Split sizes
train_size = int(len(model_df) * 0.70)
val_size = int(len(model_df) * 0.20)

# Time-based split
train_df = model_df.iloc[:train_size]
val_df = model_df.iloc[train_size:train_size + val_size]
test_df = model_df.iloc[train_size + val_size:]

# Features
X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]

# Target
y_train = train_df['is_subscription'].astype(int)
y_val = val_df['is_subscription'].astype(int)
y_test = test_df['is_subscription'].astype(int)

# Check shapes
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

# Check class distribution
print("\nTraining Distribution:")
print(y_train.value_counts())

print("\nValidation Distribution:")
print(y_val.value_counts())

print("\nTesting Distribution:")
print(y_test.value_counts())

# Check time ranges
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        data['timestamp'].min(),
        "to",
        data['timestamp'].max()
    )

Training: (149338, 10)
Validation: (42668, 10)
Testing: (21335, 10)

Training Distribution:
is_subscription
0    145091
1      4247
Name: count, dtype: int64

Validation Distribution:
is_subscription
0    40636
1     2032
Name: count, dtype: int64

Testing Distribution:
is_subscription
0    20270
1     1065
Name: count, dtype: int64
Train 2023-10-15 01:16:58 to 2025-11-16 13:29:21
Validation 2025-11-16 13:29:26 to 2026-06-28 10:07:43
Test 2026-06-28 10:21:06 to 2026-10-14 23:09:35


## Performing Random Forest 

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=3,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

# Train Model
rf.fit(X_train, y_train)

# Validation Prediction
y_val_pred = rf.predict(X_val)
y_val_prob = rf.predict_proba(X_val)[:, 1]

# Evaluation
print("Random Forest Validation Results")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nAverage Precision (PR-AUC Summary):")
print(round(average_precision_score(y_val, y_val_prob), 4))

Random Forest Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9895    0.9968    0.9932     40636
           1     0.9256    0.7894    0.8521      2032

    accuracy                         0.9869     42668
   macro avg     0.9576    0.8931    0.9226     42668
weighted avg     0.9865    0.9869    0.9865     42668


Confusion Matrix:
[[40507   129]
 [  428  1604]]

Average Precision (PR-AUC Summary):
0.9276


## Performing XGBoost Classifier

In [24]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Handle class imbalance
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost Model
xgb = XGBClassifier(
    n_estimators=350,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=5,
    scale_pos_weight=scale_weight,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Training
xgb.fit(X_train, y_train)

# Validation Prediction
y_val_pred_xgb = xgb.predict(X_val)
y_val_prob_xgb = xgb.predict_proba(X_val)[:, 1]

# Evaluation
print("XGBoost Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_xgb, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_xgb))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_xgb), 4
))

XGBoost Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9942    0.9921    0.9932     40636
           1     0.8484    0.8844    0.8660      2032

    accuracy                         0.9870     42668
   macro avg     0.9213    0.9382    0.9296     42668
weighted avg     0.9873    0.9870    0.9871     42668


Confusion Matrix:
[[40315   321]
 [  235  1797]]

Average Precision:
0.9419


## Performing LightGBM

In [25]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Class Imbalance Ratio
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# LightGBM Model
lgbm = LGBMClassifier(
    n_estimators=350,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Model Training
lgbm.fit(X_train, y_train)

# Validation Predictions
y_val_pred_lgbm = lgbm.predict(X_val)
y_val_prob_lgbm = lgbm.predict_proba(X_val)[:, 1]

# Evaluation
print("LightGBM Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_lgbm, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lgbm))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_lgbm), 4
))

LightGBM Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9938    0.9941    0.9940     40636
           1     0.8813    0.8770    0.8791      2032

    accuracy                         0.9885     42668
   macro avg     0.9376    0.9355    0.9366     42668
weighted avg     0.9885    0.9885    0.9885     42668


Confusion Matrix:
[[40396   240]
 [  250  1782]]

Average Precision:
0.9446


## Performing Catboost 

In [26]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# CatBoost Model
cat = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    auto_class_weights='Balanced',
    loss_function='Logloss',
    random_seed=42,
    verbose=False,
    thread_count=-1
)

# Train Model
cat.fit(X_train, y_train)

# Validation Predictions
y_val_pred_cat = cat.predict(X_val)
y_val_prob_cat = cat.predict_proba(X_val)[:, 1]

# Evaluation
print("CatBoost Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_cat, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_cat))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_cat), 4
))

CatBoost Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9951    0.9885    0.9918     40636
           1     0.7967    0.9026    0.8463      2032

    accuracy                         0.9844     42668
   macro avg     0.8959    0.9455    0.9191     42668
weighted avg     0.9856    0.9844    0.9849     42668


Confusion Matrix:
[[40168   468]
 [  198  1834]]

Average Precision:
0.9437


## Performing Extra Tree 